#Problem Statement 1
## Part 1

In [1]:
# ========================================
# Part 1: Understanding Returns and Covariance Structure
# ========================================

# All imports

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.preprocessing import StandardScaler

pio.templates.default = "plotly_white"
try:
    pio.renderers.default = "plotly_mimetype"
except Exception:
    pass

COLOR_SEQUENCE = px.colors.qualitative.Safe

np.random.seed(42)

### Part a

In [2]:
########################################
# (a) Generate synthetic stock prices
########################################
np.random.seed(42)  # to make random numbers reproducible
N_stocks = 10
N_days = 500

# Set parameters

P0 = 100

mu = np.random.uniform(
    0.0001,
    0.0005,
    N_stocks
)

beta = np.random.uniform(
    0.5,
    1.5,
    N_stocks
)

sigma = np.random.uniform(
    0.005,
    0.02,
    N_stocks
)

# Generate common factor
# f_t ~ N(0, 0.01)

f = np.random.normal(
    0,
    0.01,
    N_days
)

# Generate prices using the DGP
# P_i,t = P_i,t-1 * exp(mu_i + beta_i * f_t + eps_i,t)

prices = np.zeros(
    (N_days, N_stocks)
)

prices[0] = P0

for t in range(1, N_days):

    eps = np.random.normal(
        0,
        sigma
    )

    prices[t] = prices[t-1] * np.exp(
        mu + beta*f[t] + eps
    )

stock_names = [
    f"Stock_{i+1}"
    for i in range(N_stocks)
]

prices_df = pd.DataFrame(
    prices,
    columns=stock_names
)

# Plot all 10 price series using Plotly.

price_plot_df = prices_df.reset_index().melt(
    id_vars="index",
    var_name="Stock",
    value_name="Price"
).rename(columns={"index": "Trading Day"})

fig = px.line(
    price_plot_df,
    x="Trading Day",
    y="Price",
    color="Stock",
    title="Synthetic Stock Prices",
    color_discrete_sequence=COLOR_SEQUENCE,
    hover_data={"Price": ":.3f"}
)
fig.update_layout(
    height=600,
    legend_title_text="Stock",
    xaxis_title="Trading Day",
    yaxis_title="Price",
)
fig.show()

display(prices_df.head())   # to ensure all stocks start from 100

########################################
# (a-alt) Same generation, but treating 0.01 as the VARIANCE of f_t instead of the std dev
# i.e. f_t ~ N(0, var=0.01) -> std dev = sqrt(0.01) = 0.1
# Same mu, beta, sigma, and seed are reused so only the factor's scale differs
########################################
np.random.seed(42)  # reset so this run is independently reproducible

# Generate common factor under the variance interpretation
# f_t ~ N(0, 0.01) where 0.01 is the variance, so scale passed to numpy must be sqrt(0.01)

f_var_interp = np.random.normal(
    0,
    np.sqrt(0.01),
    N_days
)

# Generate prices using the same DGP as above, just with the rescaled factor
# P_i,t = P_i,t-1 * exp(mu_i + beta_i * f_t + eps_i,t)

prices_var_interp = np.zeros(
    (N_days, N_stocks)
)

prices_var_interp[0] = P0

for t in range(1, N_days):

    eps = np.random.normal(
        0,
        sigma
    )

    prices_var_interp[t] = prices_var_interp[t-1] * np.exp(
        mu + beta*f_var_interp[t] + eps
    )

prices_var_interp_df = pd.DataFrame(
    prices_var_interp,
    columns=stock_names
)

# Plot all 10 price series (variance interpretation)

price_var_plot_df = prices_var_interp_df.reset_index().melt(
    id_vars="index",
    var_name="Stock",
    value_name="Price"
).rename(columns={"index": "Trading Day"})

fig = px.line(
    price_var_plot_df,
    x="Trading Day",
    y="Price",
    color="Stock",
    title="Synthetic Stock Prices (0.01 treated as variance, std dev = 0.1)",
    color_discrete_sequence=COLOR_SEQUENCE,
    hover_data={"Price": ":.3f"}
)
fig.update_layout(
    height=600,
    legend_title_text="Stock",
    xaxis_title="Trading Day",
    yaxis_title="Price",
)
fig.show()

display(prices_var_interp_df.head())   # to ensure all stocks start from 100

# Quick numeric comparison to inform the markdown discussion below
print("Std dev interpretation -> final-day price range:",
      prices_df.iloc[-1].min(), "to", prices_df.iloc[-1].max())
print("Variance interpretation -> final-day price range:",
      prices_var_interp_df.iloc[-1].min(), "to", prices_var_interp_df.iloc[-1].max())

,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Stock_6,Stock_7,Stock_8,Stock_9,Stock_10
0,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,97.934694,100.636391,100.786630,99.354544,101.033869,97.516709,100.261987,98.896156,99.480375,100.363246
2,96.469326,99.527894,100.971323,98.366555,101.638091,95.309313,100.214723,100.152013,95.619944,99.473089
3,97.138520,99.006996,100.970142,97.575202,101.556699,94.886798,100.930177,100.221333,95.841086,99.049136
4,96.191620,97.881580,100.574733,96.762377,101.505026,97.861696,101.157403,99.237563,96.941204,98.387493


,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Stock_6,Stock_7,Stock_8,Stock_9,Stock_10
0,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,100.620230,99.371885,96.932849,99.640544,98.317408,98.269207,98.439030,97.559610,98.819894,98.482594
2,104.496679,109.311060,105.474602,103.391291,102.073684,104.043609,104.131585,103.012895,105.149143,104.145663
3,110.500848,137.332262,128.455319,115.971009,112.242706,112.030977,116.193242,120.540457,121.664197,116.924525
4,110.182401,131.194497,124.481158,112.650478,109.633432,110.358141,113.257866,117.160857,120.754192,114.445481


Std dev interpretation -> final-day price range: 68.12551063757391 to 192.0459419751247
Variance interpretation -> final-day price range: 100.96870645186242 to 198.35165050025552


### Note on the interpretation of (f_t ~ N(0, 0.01))

There is a small ambiguity in the problem statement regarding the distribution of the common factor (f_t).

For the idiosyncratic noise term, the notation is written as ε_i,t ~ N(0, σ_i²) , which suggests that the second parameter of a normal distribution represents the variance. Under that interpretation, (f_t ~ N(0,0.01)) would imply a variance of (0.01) and a standard deviation of (\sqrt{0.01}=0.1).

However, the notebook skeleton places the comment `# f_t ~ N(0, 0.01)` directly above the line where `np.random.normal()` is used. Since NumPy expects the second argument (`scale`) to be the standard deviation, this suggests that (0.01) should be interpreted as the standard deviation.

To examine this issue, synthetic stock prices were generated under both interpretations. When (0.01) is treated as the variance, the common factor becomes much more volatile, leading to extremely large price swings and causing a single factor to dominate most of the variation in returns. When (0.01) is treated as the standard deviation, the resulting stock-price paths are more moderate and the balance between common-factor risk and idiosyncratic risk is more consistent with the parameter ranges specified in the assignment.

For the remainder of this assignment, we use

[
f_t ~ N(0, 0.01)
]

with **standard deviation equal to 0.01**, i.e., `np.random.normal(0, 0.01, N_days)`, as this follows the notebook skeleton directly and produces a more meaningful PCA analysis in the later sections.


In [3]:
########################################
# (a) Generate synthetic stock prices
########################################
np.random.seed(42)  # to make random numbers reproducible
N_stocks = 10
N_days = 500

# Set parameters

P0 = 100

mu = np.random.uniform(
    0.0001,
    0.0005,
    N_stocks
)

beta = np.random.uniform(
    0.5,
    1.5,
    N_stocks
)

sigma = np.random.uniform(
    0.005,
    0.02,
    N_stocks
)

# Generate common factor
# f_t ~ N(0, 0.01)

f = np.random.normal(
    0,
    0.01,
    N_days
)

# Generate prices using the DGP
# P_i,t = P_i,t-1 * exp(mu_i + beta_i * f_t + eps_i,t)

prices = np.zeros(
    (N_days, N_stocks)
)

prices[0] = P0

for t in range(1, N_days):

    eps = np.random.normal(
        0,
        sigma
    )

    prices[t] = prices[t-1] * np.exp(
        mu + beta*f[t] + eps
    )

stock_names = [
    f"Stock_{i+1}"
    for i in range(N_stocks)
]

prices_df = pd.DataFrame(
    prices,
    columns=stock_names
)

# Plot all 10 price series using Plotly.

price_plot_df = prices_df.reset_index().melt(
    id_vars="index",
    var_name="Stock",
    value_name="Price"
).rename(columns={"index": "Trading Day"})

fig = px.line(
    price_plot_df,
    x="Trading Day",
    y="Price",
    color="Stock",
    title="Synthetic Stock Prices",
    color_discrete_sequence=COLOR_SEQUENCE,
    hover_data={"Price": ":.3f"}
)
fig.update_layout(
    height=600,
    legend_title_text="Stock",
    xaxis_title="Trading Day",
    yaxis_title="Price",
)
fig.show()

prices_df.head()   # to ensure all stocks start from 100

,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Stock_6,Stock_7,Stock_8,Stock_9,Stock_10
0,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
1,97.934694,100.636391,100.786630,99.354544,101.033869,97.516709,100.261987,98.896156,99.480375,100.363246
2,96.469326,99.527894,100.971323,98.366555,101.638091,95.309313,100.214723,100.152013,95.619944,99.473089
3,97.138520,99.006996,100.970142,97.575202,101.556699,94.886798,100.930177,100.221333,95.841086,99.049136
4,96.191620,97.881580,100.574733,96.762377,101.505026,97.861696,101.157403,99.237563,96.941204,98.387493


### Observations

1. All stocks start from the same initial price (100), but their trajectories diverge over time due to differences in drift (μ), factor loadings (β), and idiosyncratic noise (σ).

2. Stock_2 (β = 1.47) and Stock_3 (β = 1.33) show stronger reactions to market movements and experience larger price swings compared to stocks with lower β values. This demonstrates that higher factor loadings increase sensitivity to the common market factor.

3. Stock_1 (β = 0.52) shows relatively weaker participation in market-wide movements and ends significantly lower than most other stocks, illustrating the effect of a smaller factor loading.

4. Despite having similar market exposure, stocks do not follow identical paths because each stock is affected by its own idiosyncratic noise term (epsilon_{i,t}).

5. The widening gap between the best-performing and worst-performing stocks over time highlights how small differences in μ, β, and σ accumulate through exponential compounding.

### Part b

In [4]:
########################################
# (b) Compute log returns
########################################

def compute_log_returns(prices):

    # returns shape:
    # (N_days-1, N_stocks)

    log_returns = np.log(
        prices[1:] / prices[:-1]
    )

    return log_returns


returns = compute_log_returns(prices)

returns_df = pd.DataFrame(
    returns,
    columns=stock_names
)

returns_df.head()
#print(f"Returns shape: {returns_df.shape}")

,Stock_1,Stock_2,Stock_3,Stock_4,Stock_5,Stock_6,Stock_7,Stock_8,Stock_9,Stock_10
0,-0.020869,0.006344,0.007836,-0.006475,0.010286,-0.025146,0.002616,-0.011100,-0.005210,0.003626
1,-0.015076,-0.011076,0.001831,-0.009994,0.005963,-0.022896,-0.000472,0.012619,-0.039579,-0.008909
2,0.006913,-0.005247,-0.000012,-0.008077,-0.000801,-0.004443,0.007114,0.000692,0.002310,-0.004271
3,-0.009796,-0.011432,-0.003924,-0.008365,-0.000509,0.030871,0.002249,-0.009864,0.011413,-0.006702
4,-0.019004,0.020557,0.007521,0.009845,0.013010,0.040948,0.017634,0.016642,0.029120,0.002443


###Why returns not prices?
PCA requires stationary data. Prices are non-stationary — they trend upward
over time, meaning their mean and variance change with time. This violates the
assumptions of PCA, which treats the data as draws from a fixed distribution.

Log returns, defined as r_t = ln(P_t / P_{t-1}), are approximately stationary, they fluctuate around a constant mean with stable variance. PCA on returns
therefore captures genuine co-movement structure rather than spurious
correlations driven by shared price trends.

### Part c

In [5]:
########################################
# (c) Correlation and Covariance matrices
########################################

# Compute correlation matrix

corr_matrix = returns_df.corr()

# Compute covariance matrix

cov_matrix = returns_df.cov()

In [6]:
# Correlation heatmap using Plotly
fig = px.imshow(
    corr_matrix,
    x=corr_matrix.columns,
    y=corr_matrix.index,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix of Daily Log Returns"
)
fig.update_layout(height=650, width=750, xaxis_title="", yaxis_title="")
fig.show()

In [7]:
# Covariance heatmap using Plotly
fig = px.imshow(
    cov_matrix,
    x=cov_matrix.columns,
    y=cov_matrix.index,
    text_auto=".5f",
    color_continuous_scale="Viridis",
    title="Covariance Matrix of Daily Log Returns"
)
fig.update_layout(height=650, width=750, xaxis_title="", yaxis_title="")
fig.show()

In [8]:
# Highest & Lowest Correlation Pair
corr_copy = corr_matrix.copy()

np.fill_diagonal(
    corr_copy.values,
    np.nan
)

max_pair = corr_copy.stack().idxmax()
max_value = corr_copy.stack().max()

min_pair = corr_copy.stack().idxmin()
min_value = corr_copy.stack().min()

print("Highest Correlation Pair:")
print(max_pair)
print("Correlation =", round(max_value,4))

print()

print("Lowest Correlation Pair:")
print(min_pair)
print("Correlation =", round(min_value,4))

Highest Correlation Pair:
('Stock_2', 'Stock_3')
Correlation = 0.7586

Lowest Correlation Pair:
('Stock_1', 'Stock_4')
Correlation = 0.1484


### Observations
1. All correlations are positive, which is expected — all stocks share the same common market factor f_t, so they tend to move together.
2. Stock_2 and Stock_3 have the highest correlation (0.76), suggesting they have similar factor loadings β, meaning the common factor affects them almost equally.
3. Stock_1 and Stock_4 have the lowest correlation (0.15), indicating their returns are driven more by idiosyncratic noise than by the shared factor.
4. The covariance matrix shows Stock_6 has the highest variance (brightest diagonal element), consistent with it having the highest σ drawn from U(0.005, 0.02).


### Part d

In [9]:
########################################
# (d) Standardize returns
########################################

def standardize_returns(returns):

    scaler = StandardScaler()

    standardized_returns = scaler.fit_transform(
        returns
    )

    return standardized_returns, scaler


standardized_returns, scaler = standardize_returns(
    returns
)

print("Mean of standardized returns:")
print(np.mean(
    standardized_returns,
    axis=0
))

print()

print("Standard deviation of standardized returns:")
print(np.std(
    standardized_returns,
    axis=0
))

Mean of standardized returns:
[-8.45460420e-18 -2.22489584e-17 -4.53878752e-17 -6.27420627e-17
 -2.00240626e-17  3.45484607e-17  3.60433126e-17 -4.44979168e-19
  8.67709378e-18  3.02585834e-17]

Standard deviation of standardized returns:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [10]:
standardized_df = pd.DataFrame(
    standardized_returns,
    columns=stock_names
)

standardized_plot_df = standardized_df.melt(
    var_name="Stock",
    value_name="Standardized Return"
)

fig = px.box(
    standardized_plot_df,
    x="Stock",
    y="Standardized Return",
    color="Stock",
    points="outliers",
    title="Standardized Returns by Stock",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=550, showlegend=False)
fig.show()

### Observations from Standardized Returns

1. The median return of each stock is close to zero, confirming successful mean centering.
2. The spreads of the boxplots are similar across all stocks, indicating that the returns have been scaled to unit variance.
3. A few extreme observations remain visible as outliers, showing that standardization rescales the data but does not remove unusual return events.


### Why standardization matters

Stocks can have significantly different volatilities.

Without standardization, highly volatile stocks dominate the covariance matrix and therefore dominate the principal components.

Standardization transforms each stock's return series to zero mean and unit variance, ensuring all stocks contribute equally to PCA.

If standardization is skipped:

- High-volatility stocks receive larger weights.
- PCA becomes volatility-driven.
- Important correlation structures may be obscured.

### Part e
#### Observations
1. The synthetic data successfully captures both systematic and idiosyncratic risk.
2. Price non-stationarity confirmed as raw prices drifted significantly from P₀ = 100 over 500 days due to compounding drift (μ) and factor exposure (β). This confirms why PCA must be applied to log returns rather than prices directly.
3. The correlation matrix reveals the common market factor through widespread positive correlations.
4. Standardization prevents high-volatility stocks from dominating the analysis.
5. The dataset resembles realistic financial markets where common factors drive asset co-movement while firm-specific noise introduces dispersion.


## Part 2

In [11]:
# ========================================
# Part 2: Understanding PCA and Eigendecomposition
# ========================================

from sklearn.decomposition import PCA

### Part 2 a

#### PCA function

In [12]:
########################################
# (a) PCA on standardized returns
########################################

def perform_pca(data, n_components=None):

    pca = PCA(n_components=n_components)

    pca.fit(data)

    eigenvalues = pca.explained_variance_

    eigenvectors = pca.components_

    explained_var_ratio = pca.explained_variance_ratio_

    return pca, eigenvalues, eigenvectors, explained_var_ratio


pca, eigenvalues, eigenvectors, explained_var_ratio = perform_pca(
    standardized_returns
)

#### Variance table

In [13]:
variance_table = pd.DataFrame({
    "PC": range(1, len(eigenvalues)+1),
    "Eigenvalue": eigenvalues,
    "Explained Variance Ratio": explained_var_ratio,
    "Cumulative Variance": np.cumsum(explained_var_ratio)
})

variance_table

,PC,Eigenvalue,Explained Variance Ratio,Cumulative Variance
0,1,4.657302,0.464797,0.464797
1,2,0.933417,0.093155,0.557951
2,3,0.867779,0.086604,0.644555
3,4,0.731315,0.072985,0.717540
4,5,0.703557,0.070215,0.787755
5,6,0.585750,0.058458,0.846213
6,7,0.525421,0.052437,0.898650
7,8,0.447737,0.044684,0.943334
8,9,0.351836,0.035113,0.978447
9,10,0.215966,0.021553,1.000000


#### Scree plot

In [14]:
scree_df = pd.DataFrame({
    "Principal Component": np.arange(1, len(explained_var_ratio) + 1),
    "Explained Variance Ratio": explained_var_ratio
})

fig = px.line(
    scree_df,
    x="Principal Component",
    y="Explained Variance Ratio",
    markers=True,
    title="Scree Plot: PCA on Standardized Returns",
    hover_data={"Explained Variance Ratio": ":.3f"}
)
fig.update_layout(
    height=500,
    xaxis=dict(dtick=1),
    yaxis_tickformat=".1%",
)
fig.show()

#### Cumulative Variance Plot

In [15]:
cum_var = np.cumsum(explained_var_ratio)

cum_var_df = pd.DataFrame({
    "Number of Components": np.arange(1, len(cum_var) + 1),
    "Cumulative Explained Variance": cum_var
})

fig = px.line(
    cum_var_df,
    x="Number of Components",
    y="Cumulative Explained Variance",
    markers=True,
    title="Cumulative Variance Explained"
)
fig.add_hline(
    y=0.80,
    line_dash="dash",
    annotation_text="80% target",
    annotation_position="bottom right"
)
fig.update_layout(
    height=500,
    xaxis=dict(dtick=1),
    yaxis_tickformat=".1%",
)
fig.show()

Components Needed to explain ≥ 80%

In [16]:
n_components_80 = np.argmax(
    cum_var >= 0.80
) + 1

print(
    f"Components required for >=80% variance = {n_components_80}"
)

Components required for >=80% variance = 6


#### Observations

1. The first principal component (PC1) explains approximately 46.48% of the total variance, making it the most important component in the dataset.

2. There is a sharp drop in explained variance from PC1 to PC2, indicating that a dominant common factor drives a large portion of stock return variation.

3. The scree plot shows a clear elbow from PC1 to PC2 (46.5% to 9.3%), suggesting diminishing returns from adding additional principal components.

4. The cumulative variance plot shows that 6 principal components are sufficient to explain more than 80% of the total variance (84.62%).

5. Since only 6 out of 10 components capture most of the information, the return matrix exhibits substantial dimensionality reduction potential.

6. The presence of a dominant first component is consistent with the synthetic data generation process, where all stocks are influenced by a common market factor.

Part 2 b

In [17]:
########################################
# (b) PCA on covariance matrix
########################################

pca_raw, eigenvalues_raw, eigenvectors_raw, explained_var_ratio_raw = perform_pca(
    returns
)

In [18]:
eigenvalue_comparison = pd.DataFrame({
    "PC": range(1,11),
    "Eigenvalue (Standardized)": eigenvalues,
    "Eigenvalue (Raw Returns)": eigenvalues_raw
})

eigenvalue_comparison

,PC,Eigenvalue (Standardized),Eigenvalue (Raw Returns)
0,1,4.657302,0.000986
1,2,0.933417,0.000306
2,3,0.867779,0.000205
3,4,0.731315,0.000196
4,5,0.703557,0.000137
5,6,0.585750,0.000123
6,7,0.525421,0.000092
7,8,0.447737,0.000063
8,9,0.351836,0.000058
9,10,0.215966,0.000038


In [19]:
eigen_comp_df = pd.DataFrame({
    "PC": np.arange(1, 11),
    "Standardized Returns": eigenvalues,
    "Raw Returns": eigenvalues_raw
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Standardized Returns", "Raw Returns")
)

fig.add_trace(
    go.Scatter(
        x=eigen_comp_df["PC"],
        y=eigen_comp_df["Standardized Returns"],
        mode="lines+markers",
        name="Standardized eigenvalues"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=eigen_comp_df["PC"],
        y=eigen_comp_df["Raw Returns"],
        mode="lines+markers",
        name="Raw-return eigenvalues"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Eigenvalue Comparison: Standardized PCA vs Raw-Return PCA",
    height=500
)
fig.update_xaxes(title_text="Principal Component", dtick=1)
fig.update_yaxes(title_text="Eigenvalue")
fig.show()

Variance ratios comparison

In [20]:
comparison = pd.DataFrame({
    "Standardized PCA": explained_var_ratio,
    "Raw Returns PCA": explained_var_ratio_raw
})

comparison

,Standardized PCA,Raw Returns PCA
0,0.464797,0.447426
1,0.093155,0.138693
2,0.086604,0.092889
3,0.072985,0.089012
4,0.070215,0.062327
5,0.058458,0.055832
6,0.052437,0.041755
7,0.044684,0.028663
8,0.035113,0.026192
9,0.021553,0.017211


In [21]:
evr_comp_df = pd.DataFrame({
    "PC": np.arange(1, 11),
    "Standardized Returns": explained_var_ratio,
    "Raw Returns": explained_var_ratio_raw
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("PCA on Standardized Returns", "PCA on Raw Returns")
)

fig.add_trace(
    go.Scatter(
        x=evr_comp_df["PC"],
        y=evr_comp_df["Standardized Returns"],
        mode="lines+markers",
        name="Standardized PCA"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=evr_comp_df["PC"],
        y=evr_comp_df["Raw Returns"],
        mode="lines+markers",
        name="Raw-return PCA"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Explained Variance Ratio Comparison",
    height=500
)
fig.update_xaxes(title_text="Principal Component", dtick=1)
fig.update_yaxes(title_text="Explained Variance Ratio", tickformat=".1%")
fig.show()

### Comparison of Eigenvalues and Explained Variance Ratios

1. The eigenvalues obtained from standardized returns are different from those obtained from raw returns because standardization rescales each stock to unit variance.

2. In the standardized case, the total variance is approximately equal to the number of stocks (10), whereas in the raw-return case the total variance depends on the original stock volatilities.

3. PCA on raw returns gives greater weight to high-volatility stocks, causing the leading eigenvalues to be more heavily influenced by those stocks.

4. Although the overall shape of the scree plots may be similar, the magnitudes of the eigenvalues and the explained variance ratios differ because the underlying variance structure has changed.

5. This demonstrates why standardization is often preferred when stocks have substantially different volatilities.

Part 2c
loadings function

In [22]:
# Interpret loadings

def plot_loadings(
    eigenvectors,
    component_idx,
    stock_names
):

    loading_df = pd.DataFrame({
        "Stock": stock_names,
        "Loading": eigenvectors[component_idx]
    })

    fig = px.bar(
        loading_df,
        x="Stock",
        y="Loading",
        color="Loading",
        color_continuous_scale="RdBu",
        title=f"Loadings of PC{component_idx+1}"
    )
    fig.add_hline(y=0, line_width=1)
    fig.update_layout(height=500, showlegend=False)
    fig.show()

In [23]:
plot_loadings(
    eigenvectors,
    0,
    stock_names
)

In [24]:
plot_loadings(
    eigenvectors,
    1,
    stock_names
)

In [25]:
loadings_df = pd.DataFrame(
    eigenvectors.T,
    columns=[f"PC{i+1}" for i in range(10)],
    index=stock_names
)

loadings_df.round(3)

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
Stock_1,0.193,0.582,0.721,-0.154,-0.050,0.173,-0.196,0.098,0.006,-0.000
Stock_2,0.413,-0.070,0.041,0.036,0.021,-0.078,0.159,-0.180,0.225,0.841
Stock_3,0.389,-0.085,0.043,-0.124,-0.030,-0.048,0.283,-0.182,0.693,-0.477
Stock_4,0.300,-0.276,-0.182,-0.357,-0.035,0.791,-0.177,0.062,-0.110,0.004
Stock_5,0.276,0.062,-0.068,0.872,-0.124,0.242,-0.255,-0.040,0.041,-0.111
Stock_6,0.181,0.676,-0.629,-0.163,-0.285,-0.061,0.037,0.037,-0.021,0.013
Stock_7,0.355,-0.068,0.070,0.118,0.033,-0.054,0.571,0.646,-0.312,-0.080
Stock_8,0.310,-0.305,0.018,-0.160,-0.412,-0.460,-0.561,0.296,-0.007,-0.033
Stock_9,0.284,0.109,-0.175,-0.041,0.852,-0.172,-0.324,0.099,-0.008,-0.072
Stock_10,0.373,-0.051,0.090,-0.059,-0.044,-0.174,0.113,-0.635,-0.598,-0.202


### Observations
### Interpretation of Principal Component Loadings

#### PC1

1. All PC1 loadings are positive, indicating that all stocks move in the same direction when this component changes.
2. Stocks with larger PC1 loadings contribute more strongly to the common market movement.
3. This matches because all stocks were driven by the common factor:

#### PC2

1. PC2 contains both positive and negative loadings, indicating opposing behaviour among groups of stocks.

2. Stocks 1 and 6 have large positive loadings, whereas Stocks 4 and 8 have large negative loadings.

3. This suggests that PC2 captures relative performance differences rather than overall market movements.

4. Unlike PC1, PC2 represents a cross-sectional factor that separates stocks into groups with opposite sensitivities.

Part d

In [26]:
########################################
# (d) Compute and plot scores
########################################

def compute_scores(
    standardized_returns,
    eigenvectors
):

    scores = standardized_returns @ eigenvectors.T

    return scores


scores = compute_scores(
    standardized_returns,
    eigenvectors
)

print(scores.shape)

(499, 10)


In [27]:
pc1_score_df = pd.DataFrame({
    "Day": np.arange(len(scores[:, 0])),
    "PC1 Score": scores[:, 0]
})

fig = px.line(
    pc1_score_df,
    x="Day",
    y="PC1 Score",
    title="PC1 Scores Over Time",
)
fig.update_layout(height=500, xaxis_title="Day", yaxis_title="PC1 Score")
fig.show()

In [28]:
market_return = returns_df.mean(axis=1)

corr = np.corrcoef(
    scores[:,0],
    market_return
)[0,1]

print(
    f"Correlation between PC1 and Market Return = {corr:.4f}"
)

Correlation between PC1 and Market Return = 0.9891


### Observations
### Interpretation of Principal Component Scores

1. The PC1 scores fluctuate around zero over time, which is expected because the returns were standardized before applying PCA.

2. Large positive PC1 scores correspond to periods when most stocks move together in a positive direction, while large negative scores correspond to periods of broad market declines.

3. The correlation between PC1 scores and the average market return is 0.9891, indicating an almost perfect relationship.

4. This extremely high correlation confirms that PC1 represents the common market factor driving the synthetic stock returns.


In [29]:
########################################
# (e) Disprove "loadings = scores"
########################################

print("Shape of PC1 Loadings:")
print(eigenvectors[0].shape)

print()

print("Shape of PC1 Scores:")
print(scores[:,0].shape)

Shape of PC1 Loadings:
(10,)

Shape of PC1 Scores:
(499,)


In [30]:
comparison_table = pd.DataFrame({
    "Quantity": ["Loadings", "Scores"],
    "Shape": [
        str(eigenvectors[0].shape),
        str(scores[:,0].shape)
    ],
    "Meaning": [
        "Contribution of each stock to PC1",
        "Value of PC1 over time"
    ]
})

comparison_table

,Quantity,Shape,Meaning
0,Loadings,"(10,)",Contribution of each stock to PC1
1,Scores,"(499,)",Value of PC1 over time


### Why Loadings and Scores Are Different

1. Loadings measure the contribution of each stock to a principal component.

2. Scores measure the value of that principal component at each point in time.

3. For PC1, the loadings vector contains one value for each stock and therefore has shape (10,).

4. The score series contains one value for each observation day and therefore has shape (499,).

5. Loadings provide a cross-sectional view of stock contributions, whereas scores provide a time-series view of factor behaviour.

### Part 2 f
### Overall Observations

1. PCA successfully reduced the dimensionality of the return data while retaining most of the information.

2. The first principal component (PC1) explains approximately 46.48% of the total variance, making it the dominant factor in the dataset.

3. Six principal components are sufficient to explain more than 80% of the total variance, indicating that the 10-dimensional return space can be represented using a much smaller number of factors.

4. PCA performed on standardized returns and raw returns produced similar scree plot shapes because the synthetic stocks have relatively similar volatilities. However, the eigenvalues differ in magnitude due to the effect of standardization.

5. All PC1 loadings are positive, indicating that PC1 represents a common market-wide factor that affects all stocks in the same direction.

6. PC2 contains both positive and negative loadings, suggesting that it captures relative performance differences between groups of stocks rather than overall market movements.

7. The correlation between PC1 scores and the average market return is 0.9891, providing strong evidence that PC1 corresponds to the underlying market factor used in the data generation process.

8. Loadings and scores serve different purposes: loadings describe how strongly each stock contributes to a principal component, whereas scores describe how that component evolves over time.

9. The results demonstrate that PCA is an effective tool for identifying common risk factors and reducing the complexity of financial datasets.


Part 3 a


In [31]:
########################################
# (a) Reconstruct returns using k components
########################################

def reconstruct(scores, loadings, k):

    X_hat = scores[:, :k] @ loadings[:k, :]

    return X_hat


def compute_msre(X_original, X_reconstructed):

    msre = np.mean(
        (X_original - X_reconstructed) ** 2
    )

    return msre

In [32]:
msre_values = []

for k in range(1, 11):

    X_hat = reconstruct(
        scores,
        eigenvectors,
        k
    )

    msre = compute_msre(
        standardized_returns,
        X_hat
    )

    msre_values.append(msre)

msre_df = pd.DataFrame({
    "Components": range(1,11),
    "MSRE": msre_values
})

msre_df

,Components,MSRE
0,1,5.352031e-01
1,2,4.420485e-01
2,3,3.554445e-01
3,4,2.824596e-01
4,5,2.122448e-01
5,6,1.537872e-01
6,7,1.013504e-01
7,8,5.666636e-02
8,9,2.155329e-02
9,10,3.450176e-31


In [33]:
fig = px.line(
    msre_df,
    x="Components",
    y="MSRE",
    markers=True,
    title="Reconstruction Error vs Number of Components"
)
fig.update_layout(
    height=500,
    xaxis=dict(dtick=1),
    xaxis_title="Number of Components (k)",
    yaxis_title="Mean Squared Reconstruction Error"
)
fig.show()

### Observations

1. The reconstruction error decreases steadily as the number of principal components increases.

2. Using only one principal component results in a relatively high MSRE (0.5352), indicating that a single factor cannot fully explain the variation in the stock returns.

3. The first three principal components reduce the MSRE to approximately 0.3554, showing that a significant amount of information is retained even after substantial dimensionality reduction.

4. The reduction in error becomes progressively smaller as additional components are added, demonstrating diminishing marginal benefits from each new principal component.

5. The results illustrate the trade-off between compression and accuracy: fewer components provide greater dimensionality reduction but introduce larger reconstruction errors.

Part 3b

In [34]:
########################################
# (b) Per-stock reconstruction error
########################################

k = 3

X_hat_3 = reconstruct(
    scores,
    eigenvectors,
    k
)

stock_errors = np.mean(
    (standardized_returns - X_hat_3) ** 2,
    axis=0
)

error_df = pd.DataFrame({
    "Stock": stock_names,
    "Reconstruction Error": stock_errors
})

error_df.sort_values(
    by="Reconstruction Error",
    ascending=False
)

,Stock,Reconstruction Error
4,Stock_5,0.638455
8,Stock_9,0.588675
3,Stock_4,0.481467
7,Stock_8,0.466063
6,Stock_7,0.405345
9,Stock_10,0.342836
2,Stock_3,0.287655
1,Stock_2,0.202765
5,Stock_6,0.080148
0,Stock_1,0.061036


In [35]:
fig = px.bar(
    error_df,
    x="Stock",
    y="Reconstruction Error",
    color="Reconstruction Error",
    color_continuous_scale="Plasma",
    title="Per-Stock Reconstruction Error (k=3)"
)
fig.update_layout(height=500, showlegend=False)
fig.show()

In [36]:
worst_stock = stock_names[
    np.argmax(stock_errors)
]

worst_error = np.max(stock_errors)

print(
    f"Most idiosyncratic stock: {worst_stock}"
)

print(
    f"Reconstruction error: {worst_error:.4f}"
)

Most idiosyncratic stock: Stock_5
Reconstruction error: 0.6385


### Observations

1. Reconstruction errors vary significantly across stocks, indicating that the first three principal components do not explain all stocks equally well.

2. Stock_5 exhibits the highest reconstruction error (0.6385), suggesting that it is the most idiosyncratic.

3. Stocks with lower reconstruction errors are more strongly aligned with systematic market movements, whereas stocks with higher errors are influenced more heavily by idiosyncratic effects.

4. From a portfolio perspective, highly idiosyncratic stocks may contribute additional diversification benefits because their behaviour is less explained by common market factors.

### Additional reconstruction ablation: PCA subspace vs random subspaces

To check that the reconstruction results are not just an artefact of using any low-dimensional projection, the next cell compares PCA reconstruction error with random orthonormal 3-dimensional subspaces. PCA should win because it is the rank-`k` orthogonal projection that minimizes squared reconstruction error.

In [37]:
# Ablation: compare PCA k=3 reconstruction with random 3-dimensional orthonormal subspaces.
rng = np.random.default_rng(2026)
k_ablation = 3
n_random_subspaces = 250

pca_k3_msre = compute_msre(
    standardized_returns,
    reconstruct(scores, eigenvectors, k_ablation)
)

random_msre_values = []

for _ in range(n_random_subspaces):
    random_matrix = rng.normal(size=(N_stocks, k_ablation))
    q_random, _ = np.linalg.qr(random_matrix)
    X_random_hat = standardized_returns @ q_random @ q_random.T
    random_msre_values.append(compute_msre(standardized_returns, X_random_hat))

random_ablation_df = pd.DataFrame({
    "Method": ["PCA k=3"] + ["Random 3D subspace"] * n_random_subspaces,
    "MSRE": [pca_k3_msre] + random_msre_values
})

fig = px.histogram(
    random_ablation_df[random_ablation_df["Method"] == "Random 3D subspace"],
    x="MSRE",
    nbins=35,
    title="Ablation: PCA Reconstruction Error vs Random 3D Subspaces"
)
fig.add_vline(
    x=pca_k3_msre,
    line_dash="dash",
    annotation_text=f"PCA k=3 MSRE = {pca_k3_msre:.4f}",
    annotation_position="top right"
)
fig.update_layout(height=500, xaxis_title="MSRE", yaxis_title="Number of random subspaces")
fig.show()

print(f"PCA k=3 MSRE: {pca_k3_msre:.6f}")
print(f"Median random-subspace MSRE: {np.median(random_msre_values):.6f}")
print(f"Best random-subspace MSRE: {np.min(random_msre_values):.6f}")

PCA k=3 MSRE: 0.355445
Median random-subspace MSRE: 0.704470
Best random-subspace MSRE: 0.498347


Part 3c

In [38]:
########################################
# (c) Full reconstruction
########################################

X_hat_full = reconstruct(
    scores,
    eigenvectors,
    10
)

full_error = compute_msre(
    standardized_returns,
    X_hat_full
)

print(
    f"Full reconstruction MSRE = {full_error:.12f}"
)

Full reconstruction MSRE = 0.000000000000


### Why is the Error Approximately Zero?

1. PCA produces an orthonormal basis for the data space.

2. Using all 10 principal components preserves every direction of variation in the original dataset.

3. Therefore no information is discarded during reconstruction.

4. Any remaining error is due only to floating-point numerical precision.

Part 3 d

In [39]:
########################################
# (d) Correlation structure comparison
########################################

reconstructed_returns = reconstruct(
    scores,
    eigenvectors,
    3
)

corr_original = pd.DataFrame(
    standardized_returns,
    columns=stock_names
).corr()

corr_reconstructed = pd.DataFrame(
    reconstructed_returns,
    columns=stock_names
).corr()

In [40]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Original Correlation Matrix", "Reconstructed Correlation Matrix (k=3)")
)

fig.add_trace(
    go.Heatmap(
        z=corr_original.values,
        x=corr_original.columns,
        y=corr_original.index,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Corr", x=0.46)
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Heatmap(
        z=corr_reconstructed.values,
        x=corr_reconstructed.columns,
        y=corr_reconstructed.index,
        colorscale="RdBu",
        zmid=0,
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Corr", x=1.02)
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Original vs Reconstructed Correlation Structure",
    height=600,
    width=1100
)
fig.show()

In [41]:
corr_diff = np.abs(
    corr_original - corr_reconstructed
)

print(
    "Average Correlation Difference:",
    corr_diff.values.mean()
)

Average Correlation Difference: 0.3027631722310542


### Comparison of Original and Reconstructed Correlation Matrices

1. The reconstructed correlation matrix using only the first three principal components preserves the broad correlation structure of the original dataset.

2. Stocks that were positively correlated in the original matrix generally remain positively correlated after reconstruction.

3. The reconstructed correlations are noticeably stronger and more uniform than those in the original matrix.

4. Many correlation values move closer to 1 in the reconstructed matrix because PCA reconstruction using only a few components retains common systematic factors while filtering out stock-specific noise.

5. The original matrix contains both systematic and idiosyncratic effects, whereas the reconstructed matrix mainly reflects the dominant common factors captured by the first three principal components.


### Part 3e
### Observations

1. PCA provides an effective method for dimensionality reduction by representing the original return matrix using a smaller number of principal components.

2. The reconstruction error decreases monotonically as additional principal components are included, confirming that more information is retained as the dimensionality increases.

3. When all ten principal components are used, the reconstruction error becomes effectively zero, demonstrating that PCA preserves all information when the complete basis is retained.

4. The reconstructed correlation matrix preserves the major relationships among stocks but removes a portion of the idiosyncratic noise, resulting in stronger and more uniform correlations.

5. The average correlation difference of approximately 0.303 indicates that some information is lost during compression; however, the most important market structure remains intact.

6. Overall, PCA successfully captures the dominant market factors while filtering out a significant portion of stock-specific noise, making it a useful tool for financial modeling and risk analysis.

# Problem Statement 1: Part 4  
## Understanding Information Leakage in Unsupervised Learning

**PCA is unsupervised, but it can still leak future information if we fit it using test-period data.** We compare two workflows:

1. **Correct workflow:** Fit PCA only on the training period, then project the test period onto those training eigenvectors.
2. **Leaky workflow:** Fit PCA on the full dataset, including the test period, and then evaluate on the test period.

The leaky workflow is invalid because the PCA loadings and centering mean have already seen the test-period covariance structure.

In [42]:
# ========================================
# Part 4: Understanding Information Leakage in Unsupervised Learning
# ========================================

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown

np.random.seed(42)

try:
    standardized_returns
    print("Using existing standardized_returns from earlier notebook cells.")
except NameError:
    print("standardized_returns was not found, so fallback synthetic data is being generated.")

    N_stocks = 10
    N_days = 500
    P0 = 100

    mu = np.random.uniform(0.0001, 0.0005, N_stocks)
    beta = np.random.uniform(0.5, 1.5, N_stocks)
    sigma = np.random.uniform(0.005, 0.02, N_stocks)

    # Common factor and idiosyncratic shocks.
    f_t = np.random.normal(0, np.sqrt(0.01), N_days - 1)
    eps = np.random.normal(0, sigma, size=(N_days - 1, N_stocks))

    prices = np.zeros((N_days, N_stocks))
    prices[0, :] = P0

    for t in range(1, N_days):
        log_growth = mu + beta * f_t[t - 1] + eps[t - 1, :]
        prices[t, :] = prices[t - 1, :] * np.exp(log_growth)

    returns = np.log(prices[1:, :] / prices[:-1, :])
    standardized_returns = StandardScaler().fit_transform(returns)

# Convert to numpy array in case the earlier notebook stored it as a DataFrame.
standardized_returns = np.asarray(standardized_returns, dtype=float)

# Basic validation checks to avoid silent mistakes.
if standardized_returns.ndim != 2:
    raise ValueError("standardized_returns must be a 2D matrix of shape (days, stocks).")

if standardized_returns.shape[0] < 351:
    raise ValueError("standardized_returns must have at least 351 rows so that train/test split works.")

if np.isnan(standardized_returns).any():
    raise ValueError("standardized_returns contains NaN values. Please clean the earlier return matrix first.")

n_obs, n_assets = standardized_returns.shape
stock_names = [f"Stock {i+1}" for i in range(n_assets)]

print("standardized_returns shape:", standardized_returns.shape)
print("Number of observations:", n_obs)
print("Number of stocks/assets:", n_assets)

Using existing standardized_returns from earlier notebook cells.
standardized_returns shape: (499, 10)
Number of observations: 499
Number of stocks/assets: 10


---

## Part 4(a): Temporal train/test split and PCA on train only

We split the standardized return matrix chronologically:

- **Train set:** first 350 return observations
- **Test set:** all remaining observations, which should be 149 rows if the original price series had 500 days and therefore 499 log-return observations

The split must be temporal, not random. In finance, random splitting mixes future observations into the training set and gives an unrealistic estimate of model performance.

We then fit PCA only on the training set. The resulting eigenvectors are the only factor directions that would have been known at the end of the training period.

In [43]:
########################################
# (a) Temporal train/test split + PCA on train only
########################################

train_returns = standardized_returns[:350, :]
test_returns = standardized_returns[350:, :]

print("Train shape:", train_returns.shape)
print("Test shape :", test_returns.shape)

# Fit PCA on training data only.
# sklearn PCA automatically subtracts the training mean before fitting.
pca_train = PCA()
pca_train.fit(train_returns)

# Eigenvectors/loadings are stored as rows in sklearn: components_[pc_index, asset_index].
train_loadings = pca_train.components_
train_eigenvalues = pca_train.explained_variance_
train_explained_var_ratio = pca_train.explained_variance_ratio_
train_mean = pca_train.mean_

# Table of all components.
train_pca_summary = pd.DataFrame({
    "Component": np.arange(1, n_assets + 1),
    "Train eigenvalue": train_eigenvalues,
    "Train explained variance ratio": train_explained_var_ratio,
    "Train cumulative explained variance": np.cumsum(train_explained_var_ratio)
})

display(train_pca_summary)

# Display first 3 loading vectors in table form.
loading_table = pd.DataFrame(
    train_loadings[:3, :].T,
    index=stock_names,
    columns=["PC1 loading", "PC2 loading", "PC3 loading"]
)

display(loading_table)

Train shape: (350, 10)
Test shape : (149, 10)


,Component,Train eigenvalue,Train explained variance ratio,Train cumulative explained variance
0,1,4.312983,0.441717,0.441717
1,2,1.021774,0.104646,0.546363
2,3,0.886961,0.090839,0.637202
3,4,0.745505,0.076351,0.713553
4,5,0.676754,0.069310,0.782863
5,6,0.556236,0.056967,0.839831
6,7,0.548531,0.056178,0.896009
7,8,0.459199,0.047029,0.943038
8,9,0.347605,0.035600,0.978638
9,10,0.208578,0.021362,1.000000


,PC1 loading,PC2 loading,PC3 loading
Stock 1,0.180603,0.399805,0.812295
Stock 2,0.413409,-0.049998,0.037465
Stock 3,0.389973,-0.110905,-0.032964
Stock 4,0.284341,-0.269516,-0.197606
Stock 5,0.270533,0.151680,-0.000485
Stock 6,0.185169,0.742544,-0.506089
Stock 7,0.378292,-0.020612,0.086195
Stock 8,0.292596,-0.392531,-0.063766
Stock 9,0.291206,0.136448,-0.154686
Stock 10,0.377562,-0.072217,0.084200


### Explanation for Part 4(a)

The PCA model above is the **honest training-period factor model**. It has only seen the first 350 observations.

The columns in the summary table mean:

- **Eigenvalue:** absolute variance captured by that principal component inside the training period.
- **Explained variance ratio:** fraction of training variance captured by that component.
- **Cumulative explained variance:** total variance captured by the first `k` components.

The loading table gives the asset weights inside the first few principal components. These loadings are the factor directions that will be used to project the test data in the next section.

---

## Part 4(b): Project the test data onto the training eigenvectors

For the correct out-of-sample procedure, the test data must be transformed using only training-period information.

That means:

1. Center test returns using the **training mean** learned by `pca_train`.
2. Project the centered test data onto the **training eigenvectors**.
3. Reconstruct the test data using only the first `k` components.
4. Compute explained variance using:

$$
EV_{\text{test}}(k)
=
1 -
\frac{\left\lVert X_{\text{test}} - \widehat{X}_{\text{test},k} \right\rVert_F^2}
{\left\lVert X_{\text{test}} \right\rVert_F^2}
$$

Here, $\left\lVert \cdot \right\rVert_F$ is the Frobenius norm. Intuitively, this formula asks: **how much of the test-period variation can the train-fitted PCA subspace explain?**


In [44]:
########################################
# (b) Project test data onto TRAIN eigenvectors
########################################

def compute_explained_variance_oos(test_data, train_loadings, k, train_mean):
    """
    Compute out-of-sample explained variance for test_data using PCA loadings
    learned only from the training period.
    """

    # Center test data using training mean only.
    # Using test_data.mean(axis=0) would leak test statistics.
    X_test_centered = test_data - train_mean

    # Select first k eigenvectors.
    V_k = train_loadings[:k, :]      # shape: (k, n_assets)

    # Project test observations onto training PCs.
    scores_test = X_test_centered @ V_k.T     # shape: (n_test, k)

    # Reconstruct centered test observations using the selected PCs.
    reconstructed_test = scores_test @ V_k    # shape: (n_test, n_assets)

    # Residual = what the selected PCA subspace fails to explain.
    residual_test = X_test_centered - reconstructed_test

    # Explained variance = 1 - residual variance / total variance.
    total_variance = np.sum(X_test_centered ** 2)
    residual_variance = np.sum(residual_test ** 2)
    explained_var_ratio_test = 1 - residual_variance / total_variance

    return explained_var_ratio_test, reconstructed_test, residual_test

# Compare train vs clean out-of-sample test explained variance for k = 1, 2, 3.
clean_rows = []
for k in [1, 2, 3]:
    ev_test_clean, _, residual_clean = compute_explained_variance_oos(
        test_data=test_returns,
        train_loadings=train_loadings,
        k=k,
        train_mean=train_mean
    )

    clean_rows.append({
        "k": k,
        "Train cumulative EV": np.sum(train_explained_var_ratio[:k]),
        "Clean test EV using train PCs": ev_test_clean,
        "Drop: train EV - clean test EV": np.sum(train_explained_var_ratio[:k]) - ev_test_clean,
        "Clean residual variance share": 1 - ev_test_clean
    })

clean_ev_df = pd.DataFrame(clean_rows)
display(clean_ev_df)

,k,Train cumulative EV,Clean test EV using train PCs,Drop: train EV - clean test EV,Clean residual variance share
0,1,0.441717,0.514113,-0.072396,0.485887
1,2,0.546363,0.579721,-0.033358,0.420279
2,3,0.637202,0.656649,-0.019447,0.343351


### Explanation for Part 4(b)

The clean test explained variance will usually be lower than the training explained variance. That drop is normal because PCA was optimized to explain the training covariance structure, not the future/test covariance structure.

Financially, this means that the factor model learned from the past does not perfectly explain future returns. The unexplained part is out-of-sample residual risk. If correlations, volatilities, or factor loadings change between the train and test periods, the drop becomes larger.

---

## Part 4(c): Leaky PCA fitted on the full dataset

Now we intentionally perform the wrong procedure:

1. Fit PCA on the **entire dataset**, which includes both train and test periods.
2. Use the full-data eigenvectors to reconstruct only the test data.
3. Compare the test explained variance from this leaky method with the clean method.

This is information leakage because the PCA loadings are influenced by the test period. Even though PCA has no labels or target variable, it still learns from the test-period covariance structure.

In [45]:
########################################
# (c) LEAKY approach: PCA on full data
########################################

# Fit PCA on all observations: train + test.
# This is the invalid/leaky step.
pca_full = PCA()
pca_full.fit(standardized_returns)

full_loadings = pca_full.components_
full_mean = pca_full.mean_
full_explained_var_ratio = pca_full.explained_variance_ratio_
full_eigenvalues = pca_full.explained_variance_

def compute_explained_variance_with_given_pca(test_data, pca_model, k):
    """
    Compute explained variance of test_data using an already-fitted PCA model.
    This function is used for the leaky full-data PCA model.
    """

    X_test_centered = test_data - pca_model.mean_
    V_k = pca_model.components_[:k, :]
    scores_test = X_test_centered @ V_k.T
    reconstructed_test = scores_test @ V_k
    residual_test = X_test_centered - reconstructed_test

    total_variance = np.sum(X_test_centered ** 2)
    residual_variance = np.sum(residual_test ** 2)
    explained_var_ratio = 1 - residual_variance / total_variance

    return explained_var_ratio, reconstructed_test, residual_test

# Compare clean and leaky test explained variance for k = 1, 2, 3.
leakage_rows = []
for k in [1, 2, 3]:
    ev_test_clean, _, _ = compute_explained_variance_oos(
        test_data=test_returns,
        train_loadings=train_loadings,
        k=k,
        train_mean=train_mean
    )

    ev_test_leaky, _, _ = compute_explained_variance_with_given_pca(
        test_data=test_returns,
        pca_model=pca_full,
        k=k
    )

    leakage_rows.append({
        "k": k,
        "Train cumulative EV (train PCA)": np.sum(train_explained_var_ratio[:k]),
        "Full-data cumulative EV (leaky PCA)": np.sum(full_explained_var_ratio[:k]),
        "Clean test EV using train PCs": ev_test_clean,
        "Leaky test EV using full-data PCs": ev_test_leaky,
        "Leakage gap: leaky - clean": ev_test_leaky - ev_test_clean,
        "Clean residual variance share": 1 - ev_test_clean,
        "Leaky residual variance share": 1 - ev_test_leaky
    })

leakage_df = pd.DataFrame(leakage_rows)
display(leakage_df)

,k,Train cumulative EV (train PCA),Full-data cumulative EV (leaky PCA),Clean test EV using train PCs,Leaky test EV using full-data PCs,Leakage gap: leaky - clean,Clean residual variance share,Leaky residual variance share
0,1,0.441717,0.464797,0.514113,0.514618,0.000505,0.485887,0.485382
1,2,0.546363,0.557951,0.579721,0.586398,0.006677,0.420279,0.413602
2,3,0.637202,0.644555,0.656649,0.663587,0.006938,0.343351,0.336413


### Explanation for Part 4(c)

The leaky method is invalid because it lets the PCA model see the full dataset before evaluation. This changes both:

- the **mean vector** used for centering, and
- the **eigenvectors/loadings** used as factor directions.

As a result, the leaky PCA basis is partially adapted to the test period. This often makes the test explained variance look better than it should. In practical terms, it can make a model appear more stable and more predictive than it really is.

The phrase **"PCA is unsupervised"** does not remove leakage. Leakage is not only about labels. Any use of future/test-period data to choose model parameters is leakage.

---

## Visual comparison: clean vs leaky explained variance

The next plots compare:

- training cumulative explained variance,
- clean test explained variance using train-only PCA,
- leaky test explained variance using full-data PCA,
- and the leakage gap.


In [46]:
# Visual comparison for k = 1, 2, 3.
plot_df = leakage_df.set_index("k")[[
    "Train cumulative EV (train PCA)",
    "Clean test EV using train PCs",
    "Leaky test EV using full-data PCs"
]].reset_index()

plot_long = plot_df.melt(
    id_vars="k",
    var_name="Procedure",
    value_name="Explained Variance Ratio"
)

fig = px.line(
    plot_long,
    x="k",
    y="Explained Variance Ratio",
    color="Procedure",
    markers=True,
    title="Train vs Clean Test vs Leaky Test Explained Variance",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(
    height=520,
    xaxis_title="Number of principal components retained (k)",
    yaxis_title="Explained variance ratio",
    yaxis_tickformat=".1%",
    xaxis=dict(dtick=1)
)
fig.show()

gap_df = leakage_df.rename(columns={"Leakage gap: leaky - clean": "Leakage gap"})
fig = px.bar(
    gap_df,
    x="k",
    y="Leakage gap",
    color="Leakage gap",
    color_continuous_scale="RdBu",
    title="Estimated Information Leakage Gap"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(
    height=480,
    xaxis_title="Number of principal components retained (k)",
    yaxis_title="Leaky EV - clean EV",
    xaxis=dict(dtick=1),
    yaxis_tickformat=".2%"
)
fig.show()

---

## Part 4(d): Quantify the leakage gap

The leakage gap is:

$$
\mathrm{Leakage\ Gap}(k)
=
EV_{\text{test, leaky}}(k)
-
EV_{\text{test, clean}}(k)
$$

A positive gap means that fitting PCA on the full dataset made test performance look better than the honest out-of-sample procedure.

The gap is expected to be largest when the test period has a different covariance/correlation structure from the training period, especially during regime changes. It is expected to be smallest when the train and test periods are generated by almost the same covariance structure.


In [47]:
########################################
# (d) Quantify the leakage gap
########################################

# Extract the k=1,2,3 gaps clearly.
gap_table = leakage_df[[
    "k",
    "Clean test EV using train PCs",
    "Leaky test EV using full-data PCs",
    "Leakage gap: leaky - clean",
    "Clean residual variance share",
    "Leaky residual variance share"
]].copy()

display(gap_table)

# Create an automatic markdown-style summary using the actual numbers from the run.
summary_lines = []
for _, row in gap_table.iterrows():
    summary_lines.append(
        f"- For k = {int(row['k'])}, clean test EV = {row['Clean test EV using train PCs']:.4f}, "
        f"leaky test EV = {row['Leaky test EV using full-data PCs']:.4f}, "
        f"gap = {row['Leakage gap: leaky - clean']:.4f}."
    )

display(Markdown("### Numerical leakage summary from this run\n" + "\n".join(summary_lines)))

,k,Clean test EV using train PCs,Leaky test EV using full-data PCs,Leakage gap: leaky - clean,Clean residual variance share,Leaky residual variance share
0,1,0.514113,0.514618,0.000505,0.485887,0.485382
1,2,0.579721,0.586398,0.006677,0.420279,0.413602
2,3,0.656649,0.663587,0.006938,0.343351,0.336413


### Numerical leakage summary from this run
- For k = 1, clean test EV = 0.5141, leaky test EV = 0.5146, gap = 0.0005.
- For k = 2, clean test EV = 0.5797, leaky test EV = 0.5864, gap = 0.0067.
- For k = 3, clean test EV = 0.6566, leaky test EV = 0.6636, gap = 0.0069.

### Explanation for Part 4(d)

The clean residual variance share is the honest estimate of how much test-period risk is left unexplained after using the PCA factors learned from the training period.

The leaky residual variance share is usually smaller because the PCA basis has already adjusted to the test period. This is dangerous in finance because it can make the desk believe that the factor model hedges more risk than it actually would in live trading.

So the practical consequence is:

> The leaky procedure usually **overstates explained variance** and **understates residual risk**.

This is exactly why unsupervised preprocessing steps such as PCA, scaling, clustering, and feature selection must be fit inside the training window only.

---

## Additional diagnostic 1: leakage gap across all components

The required comparison for `k = 1, 2, 3` is useful, but the full pattern across all available components gives a better picture. If the clean and leaky curves are close for every `k`, the train and test covariance structures are fairly stable. If the curves separate strongly for small `k`, the leaky PCA basis is adapting to test-period structure that the train-only model could not have known.


In [48]:
# Diagnostic 1: extend the leakage comparison to all k = 1, ..., n_assets.
ks_all = np.arange(1, n_assets + 1)
all_k_rows = []

for k in ks_all:
    ev_clean, _, _ = compute_explained_variance_oos(
        test_data=test_returns,
        train_loadings=train_loadings,
        k=k,
        train_mean=train_mean
    )
    ev_leaky, _, _ = compute_explained_variance_with_given_pca(
        test_data=test_returns,
        pca_model=pca_full,
        k=k
    )
    all_k_rows.append({
        "k": k,
        "Clean test EV": ev_clean,
        "Leaky test EV": ev_leaky,
        "Leakage gap": ev_leaky - ev_clean,
        "Clean residual share": 1 - ev_clean,
        "Leaky residual share": 1 - ev_leaky
    })

all_k_df = pd.DataFrame(all_k_rows)
display(all_k_df)

all_k_long = all_k_df.melt(
    id_vars="k",
    value_vars=["Clean test EV", "Leaky test EV"],
    var_name="Procedure",
    value_name="Test explained variance ratio"
)

fig = px.line(
    all_k_long,
    x="k",
    y="Test explained variance ratio",
    color="Procedure",
    markers=True,
    title="Leakage Gap Across All Components",
    color_discrete_sequence=COLOR_SEQUENCE
)

fig.add_trace(
    go.Bar(
        x=all_k_df["k"],
        y=all_k_df["Leakage gap"],
        name="Leakage gap",
        opacity=0.25,
        yaxis="y2"
    )
)

fig.update_layout(
    height=540,
    xaxis=dict(dtick=1, title="Number of retained principal components (k)"),
    yaxis=dict(title="Test explained variance ratio", tickformat=".1%"),
    yaxis2=dict(
        title="Leaky EV - clean EV",
        overlaying="y",
        side="right",
        tickformat=".2%"
    ),
    legend_title_text="Series"
)
fig.show()

fig = px.bar(
    all_k_df,
    x="k",
    y="Leakage gap",
    color="Leakage gap",
    color_continuous_scale="RdBu",
    title="Leakage Gap by Number of Components"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(
    height=460,
    xaxis=dict(dtick=1, title="Number of retained principal components (k)"),
    yaxis_title="Leaky EV - clean EV",
    yaxis_tickformat=".2%"
)
fig.show()

max_gap_row = all_k_df.iloc[all_k_df["Leakage gap"].abs().argmax()]
print(
    f"Largest absolute leakage gap occurs at k={int(max_gap_row['k'])}: "
    f"gap={max_gap_row['Leakage gap']:.6f}."
)

,k,Clean test EV,Leaky test EV,Leakage gap,Clean residual share,Leaky residual share
0,1,0.514113,0.514618,0.000505,0.485887,0.485382
1,2,0.579721,0.586398,0.006677,0.420279,0.413602
2,3,0.656649,0.663587,0.006938,0.343351,0.336413
3,4,0.712401,0.739327,0.026926,0.287599,0.260673
4,5,0.789676,0.805003,0.015327,0.210324,0.194997
5,6,0.853048,0.864471,0.011423,0.146952,0.135529
6,7,0.898817,0.908316,0.009499,0.101183,0.091684
7,8,0.942621,0.944909,0.002288,0.057379,0.055091
8,9,0.977574,0.978398,0.000825,0.022426,0.021602
9,10,1.000000,1.000000,0.000000,0.000000,0.000000


Largest absolute leakage gap occurs at k=4: gap=0.026926.


### Interpretation of the full-`k` plot

The all-component plot is a useful sanity check. At `k = n_assets`, both methods should explain essentially all centered test variation, so the gap should collapse near zero. The most meaningful leakage effect is usually at small or moderate `k`, because those are the dimensions where the choice of PCA subspace matters most.


---

## Additional diagnostic 2: eigenvector alignment between clean and leaky PCA

Explained variance is only a scalar. To see *how* leakage changes the factor model, we compare the train-only loading vectors with the full-data loading vectors.

For each component, we compute sign-invariant cosine similarity:

$$
\left|\cos(\theta_j)\right|
=
\left|
\frac{v_{j,\text{train}}^\top v_{j,\text{full}}}
{\lVert v_{j,\text{train}}\rVert_2\,\lVert v_{j,\text{full}}\rVert_2}
\right|
$$

We take the absolute value because PCA eigenvectors are sign-ambiguous: if `v` is a valid eigenvector, then `-v` is also valid.


In [49]:
# Diagnostic 2: cosine similarity between train-only and full-data eigenvectors.
def cosine_similarity_abs(v1, v2):
    """Sign-invariant cosine similarity between two loading vectors."""
    denom = np.linalg.norm(v1) * np.linalg.norm(v2)
    if denom == 0:
        return np.nan
    return abs(float(np.dot(v1, v2) / denom))

alignment_rows = []
for pc_idx in range(n_assets):
    cos_sim = cosine_similarity_abs(train_loadings[pc_idx], full_loadings[pc_idx])
    alignment_rows.append({
        "Component": pc_idx + 1,
        "Abs cosine similarity: train vs full-data loading": cos_sim,
        "Approx rotation angle (degrees)": np.degrees(np.arccos(np.clip(cos_sim, -1, 1)))
    })

alignment_df = pd.DataFrame(alignment_rows)
display(alignment_df)

for pc_idx in range(min(3, n_assets)):
    print(
        f"PC{pc_idx + 1} cosine similarity "
        f"(train-only vs full-data loadings, sign-invariant): "
        f"{alignment_df.loc[pc_idx, 'Abs cosine similarity: train vs full-data loading']:.6f}"
    )

fig = px.line(
    alignment_df,
    x="Component",
    y="Abs cosine similarity: train vs full-data loading",
    markers=True,
    title="Eigenvector Alignment: Train-only PCA vs Full-data PCA"
)
fig.add_hline(y=1.0, line_width=1, annotation_text="perfect alignment")
fig.update_layout(
    height=480,
    xaxis=dict(dtick=1),
    yaxis=dict(range=[0, 1.05], title="Absolute cosine similarity")
)
fig.show()

fig = px.bar(
    alignment_df,
    x="Component",
    y="Approx rotation angle (degrees)",
    color="Approx rotation angle (degrees)",
    color_continuous_scale="Viridis",
    title="Approximate Loading Rotation Angle by Component"
)
fig.update_layout(height=480, xaxis=dict(dtick=1))
fig.show()

,Component,Abs cosine similarity: train vs full-data loading,Approx rotation angle (degrees)
0,1,0.999309,2.129532
1,2,0.970984,13.835950
2,3,0.979253,11.691548
3,4,0.594241,53.541479
4,5,0.615378,52.020586
5,6,0.916556,23.572283
6,7,0.948953,18.385933
7,8,0.949875,18.217736
8,9,0.995289,5.563456
9,10,0.997976,3.646211


PC1 cosine similarity (train-only vs full-data loadings, sign-invariant): 0.999309
PC2 cosine similarity (train-only vs full-data loadings, sign-invariant): 0.970984
PC3 cosine similarity (train-only vs full-data loadings, sign-invariant): 0.979253


### Interpretation of eigenvector alignment

A cosine similarity close to 1 means the clean and leaky loadings point in almost the same direction. A lower value means that including the test period rotated the PCA factor direction. This is exactly the geometric form of leakage: the full-data PCA model has partly learned future covariance information.


---

## Additional diagnostic 3: per-day reconstruction residual on the test set

The scalar explained-variance ratio hides *when* the clean and leaky models differ. A residual time-series shows whether the leaky model mainly improves reconstruction during a particular cluster or regime inside the test period.


In [50]:
# Diagnostic 3: per-day squared reconstruction residual for clean vs leaky PCA.
k_residual = min(3, n_assets)

_, _, residual_clean_k = compute_explained_variance_oos(
    test_data=test_returns,
    train_loadings=train_loadings,
    k=k_residual,
    train_mean=train_mean
)
_, _, residual_leaky_k = compute_explained_variance_with_given_pca(
    test_data=test_returns,
    pca_model=pca_full,
    k=k_residual
)

daily_res_clean = np.sum(residual_clean_k ** 2, axis=1)
daily_res_leaky = np.sum(residual_leaky_k ** 2, axis=1)
residual_time_df = pd.DataFrame({
    "Test day index": np.arange(len(test_returns)),
    f"Clean residual squared, k={k_residual}": daily_res_clean,
    f"Leaky residual squared, k={k_residual}": daily_res_leaky,
    "Residual reduction from leakage": daily_res_clean - daily_res_leaky
})

display(residual_time_df.head())

residual_long = residual_time_df.melt(
    id_vars="Test day index",
    value_vars=[
        f"Clean residual squared, k={k_residual}",
        f"Leaky residual squared, k={k_residual}"
    ],
    var_name="Procedure",
    value_name="Squared reconstruction residual"
)

fig = px.line(
    residual_long,
    x="Test day index",
    y="Squared reconstruction residual",
    color="Procedure",
    title=f"Per-Day Reconstruction Residual: Clean vs Leaky PCA (k={k_residual})",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=500)
fig.show()

fig = px.line(
    residual_time_df,
    x="Test day index",
    y="Residual reduction from leakage",
    title="Where the Leaky PCA Reduces Apparent Test Residual Risk"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=500, yaxis_title="Clean residual - leaky residual")
fig.show()

print(
    f"For k={k_residual}, total clean residual share = "
    f"{1 - all_k_df.loc[all_k_df['k'] == k_residual, 'Clean test EV'].iloc[0]:.6f}."
)
print(
    f"For k={k_residual}, total leaky residual share = "
    f"{1 - all_k_df.loc[all_k_df['k'] == k_residual, 'Leaky test EV'].iloc[0]:.6f}."
)

,Test day index,"Clean residual squared, k=3","Leaky residual squared, k=3",Residual reduction from leakage
0,0,9.507960,9.271772,0.236188
1,1,2.644788,2.189242,0.455546
2,2,2.568008,2.556046,0.011962
3,3,2.488600,2.533023,-0.044423
4,4,2.653707,2.926738,-0.273031


For k=3, total clean residual share = 0.343351.
For k=3, total leaky residual share = 0.336413.


### Interpretation of the residual plot

If the leaky residual is consistently below the clean residual, the full-data PCA is making the test period look easier to explain than it would have been in a live setting. If the difference is concentrated in a few test days, that suggests the leaky PCA has adapted to a specific future episode rather than representing a stable factor model.


---

## Additional diagnostic 4: small regime-change stress test

Part 4(d) asks when the leakage gap would be largest. To demonstrate this instead of only stating it, we create a controlled stress test: the training period is left unchanged, but the test period receives an extra latent factor with an alternating-sign loading pattern. This creates a test-only covariance structure that the train-only PCA could not have learned.

This is not used as the main answer; it is only a diagnostic experiment to show why regime changes make leakage more severe.


In [51]:
# Diagnostic 4: demonstrate that the leakage gap widens under a test-only regime change.
def leakage_curve_for_dataset(data, split_idx=350):
    """Return clean-vs-leaky EV curves for a given return matrix."""
    data = np.asarray(data, dtype=float)
    train_X = data[:split_idx, :]
    test_X = data[split_idx:, :]

    pca_tr = PCA().fit(train_X)
    pca_all = PCA().fit(data)

    rows = []
    for k in range(1, data.shape[1] + 1):
        ev_clean, _, _ = compute_explained_variance_oos(
            test_data=test_X,
            train_loadings=pca_tr.components_,
            k=k,
            train_mean=pca_tr.mean_
        )
        ev_leaky, _, _ = compute_explained_variance_with_given_pca(
            test_data=test_X,
            pca_model=pca_all,
            k=k
        )
        rows.append({
            "k": k,
            "Clean test EV": ev_clean,
            "Leaky test EV": ev_leaky,
            "Leakage gap": ev_leaky - ev_clean
        })
    return pd.DataFrame(rows)

# Build a deterministic test-only regime shift.
rng = np.random.default_rng(2026)
stressed_returns = standardized_returns.copy()

# Alternating-sign direction: roughly a sector-rotation style factor.
regime_direction = np.ones(n_assets)
regime_direction[1::2] = -1
regime_direction = regime_direction / np.linalg.norm(regime_direction)

# Add a new test-only latent factor. The scale controls severity of the regime change.
regime_scale = 1.50
new_factor = rng.normal(loc=0.0, scale=regime_scale, size=(test_returns.shape[0], 1))
stressed_returns[350:, :] = stressed_returns[350:, :] + new_factor @ regime_direction.reshape(1, -1)

baseline_curve = all_k_df[["k", "Clean test EV", "Leaky test EV", "Leakage gap"]].copy()
stress_curve = leakage_curve_for_dataset(stressed_returns, split_idx=350)

stress_compare_df = baseline_curve[["k", "Leakage gap"]].rename(
    columns={"Leakage gap": "Baseline leakage gap"}
).merge(
    stress_curve[["k", "Leakage gap"]].rename(columns={"Leakage gap": "Regime-change leakage gap"}),
    on="k"
)
stress_compare_df["Gap increase from regime change"] = (
    stress_compare_df["Regime-change leakage gap"] - stress_compare_df["Baseline leakage gap"]
)

display(stress_compare_df)

stress_long = stress_compare_df.melt(
    id_vars="k",
    value_vars=["Baseline leakage gap", "Regime-change leakage gap"],
    var_name="Scenario",
    value_name="Leakage gap"
)

fig = px.line(
    stress_long,
    x="k",
    y="Leakage gap",
    color="Scenario",
    markers=True,
    title="Leakage Gap Widens When the Test Covariance Regime Changes",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(
    height=520,
    xaxis=dict(dtick=1, title="Number of retained principal components (k)"),
    yaxis_title="Leaky EV - clean EV",
    yaxis_tickformat=".2%"
)
fig.show()

fig = px.bar(
    stress_compare_df,
    x="k",
    y="Gap increase from regime change",
    color="Gap increase from regime change",
    color_continuous_scale="Plasma",
    title="Increase in Leakage Gap Caused by Test-Only Regime Change"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=480, xaxis=dict(dtick=1), yaxis_tickformat=".2%")
fig.show()

baseline_max_gap = float(stress_compare_df["Baseline leakage gap"].abs().max())
stress_max_gap = float(stress_compare_df["Regime-change leakage gap"].abs().max())

print(f"Largest absolute baseline gap      : {baseline_max_gap:.6f}")
print(f"Largest absolute regime-change gap : {stress_max_gap:.6f}")
print(f"Increase in largest absolute gap   : {stress_max_gap - baseline_max_gap:.6f}")

,k,Baseline leakage gap,Regime-change leakage gap,Gap increase from regime change
0,1,0.000505,-0.000098,-0.000603
1,2,0.006677,0.178796,0.172119
2,3,0.006938,0.131260,0.124322
3,4,0.026926,0.151717,0.124791
4,5,0.015327,0.072441,0.057114
5,6,0.011423,0.069454,0.058032
6,7,0.009499,0.062787,0.053289
7,8,0.002288,0.056284,0.053996
8,9,0.000825,0.034840,0.034016
9,10,0.000000,0.000000,0.000000


Largest absolute baseline gap      : 0.026926
Largest absolute regime-change gap : 0.178796
Increase in largest absolute gap   : 0.151870


### Interpretation of the stress test

The stress test directly supports the qualitative claim in Part 4(d): leakage is most damaging when the test period has a covariance pattern that was not present in training. In that situation, full-data PCA can rotate toward the future structure, while train-only PCA cannot. This makes the leaky result look better precisely when an honest live model would struggle the most.


---

## Bias direction and PCA optimality

For a fixed centered test matrix and a fixed `k`, a PCA reconstruction error is:

$$
\lVert X_{\text{test}} - \widehat{X}_{\text{test},k} \rVert_F^2
$$

Lower reconstruction error means higher explained variance. PCA fitted directly on the test set would choose the rank-`k` subspace that minimizes this error on the test set itself. That is the strongest possible form of leakage.

The full-data PCA used here is slightly different: it is optimized on train + test together, not on the test set alone. So in a finite sample it is safest to say this: **whenever the leaky test EV is higher than the clean test EV, the leaky procedure has overstated explained variance and understated residual risk by exactly the gap shown in the table.** The procedure is invalid regardless of the sign of the observed gap because its mean and eigenvectors were estimated using future/test-period data.


---

## Part 4(e): Final observations with run-specific numbers

The next cell prints the final observations using the actual numbers from this notebook run. This makes the conclusion stronger than a generic paragraph because the interpretation is tied directly to the measured leakage gap, eigenvector rotation, and residual-risk bias.


In [52]:
# Final observations generated from the actual run.
k_focus = min(3, n_assets)
focus_row = all_k_df.loc[all_k_df["k"] == k_focus].iloc[0]
max_gap_row = all_k_df.iloc[all_k_df["Leakage gap"].abs().argmax()]
pc1_alignment = alignment_df.loc[0, "Abs cosine similarity: train vs full-data loading"]

clean_ev_focus = float(focus_row["Clean test EV"])
leaky_ev_focus = float(focus_row["Leaky test EV"])
gap_focus = float(focus_row["Leakage gap"])
clean_resid_focus = float(focus_row["Clean residual share"])
leaky_resid_focus = float(focus_row["Leaky residual share"])
residual_bias = clean_resid_focus - leaky_resid_focus

if residual_bias >= 0:
    bias_direction_sentence = (
        f"For k={k_focus}, the leaky procedure understates residual variance by "
        f"{residual_bias:.4f} of total centered test variation, because its residual share is "
        f"{leaky_resid_focus:.4f} instead of the clean {clean_resid_focus:.4f}."
    )
else:
    bias_direction_sentence = (
        f"For k={k_focus}, this particular finite-sample run gives a negative gap. "
        f"That does not make the workflow valid; it still used future data to choose the PCA basis."
    )

observations_md = f"""
### Final observations

1. **Clean out-of-sample EV is the number that matters.** For `k={k_focus}`, the train-only PCA explains **{clean_ev_focus:.4f}** of the test-period variation, while the leaky full-data PCA explains **{leaky_ev_focus:.4f}**.

2. **The measured leakage gap is not just theoretical.** For `k={k_focus}`, the gap is **{gap_focus:.4f}**. Across all components, the largest absolute gap occurs at `k={int(max_gap_row['k'])}` with value **{float(max_gap_row['Leakage gap']):.4f}**.

3. **Leakage changes the geometry of the factor model.** The sign-invariant cosine similarity between train-only PC1 and full-data PC1 is **{pc1_alignment:.4f}**. Values below 1 mean that the full-data PCA loading vector has rotated because it was allowed to see the test period.

4. **The bias goes directly into risk perception.** {bias_direction_sentence} A risk manager using the leaky PCA would believe the selected factors explain more live risk than they honestly do.

5. **The regime-change stress test confirms the intuition.** The largest absolute baseline leakage gap is **{baseline_max_gap:.4f}**, while after injecting a test-only covariance regime change it becomes **{stress_max_gap:.4f}**. This shows that leakage is most dangerous exactly when market structure changes and an honest out-of-sample model should be treated carefully.
"""

display(Markdown(observations_md))



### Final observations

1. **Clean out-of-sample EV is the number that matters.** For `k=3`, the train-only PCA explains **0.6566** of the test-period variation, while the leaky full-data PCA explains **0.6636**.

2. **The measured leakage gap is not just theoretical.** For `k=3`, the gap is **0.0069**. Across all components, the largest absolute gap occurs at `k=4` with value **0.0269**.

3. **Leakage changes the geometry of the factor model.** The sign-invariant cosine similarity between train-only PC1 and full-data PC1 is **0.9993**. Values below 1 mean that the full-data PCA loading vector has rotated because it was allowed to see the test period.

4. **The bias goes directly into risk perception.** For k=3, the leaky procedure understates residual variance by 0.0069 of total centered test variation, because its residual share is 0.3364 instead of the clean 0.3434. A risk manager using the leaky PCA would believe the selected factors explain more live risk than they honestly do.

5. **The regime-change stress test confirms the intuition.** The largest absolute baseline leakage gap is **0.0269**, while after injecting a test-only covariance regime change it becomes **0.1788**. This shows that leakage is most dangerous exactly when market structure changes and an honest out-of-sample model should be treated carefully.


## 5. Factor-Neutral Portfolio Construction

In this section, we construct and evaluate a factor-neutral portfolio using PCA. The goal is to remove exposure to dominant market factors and analyze the resulting portfolio performance.

We proceed step-by-step by first analyzing factor exposures, then constructing the portfolio, and finally evaluating its performance.

### (a) PC1 Exposure Analysis

We begin by computing the exposure of each stock to the first principal component (PC1), which represents the dominant market factor. Stocks with high loadings on PC1 are more sensitive to overall market movements.

To construct a factor-neutral portfolio, we aim to remove this dominant exposure.

In [53]:
# Step 1: Extract PC1 loadings

pc1 = pca.components_[0]   # first principal component

import pandas as pd

pc1_df = pd.DataFrame({
    "Stock": [f"Stock{i+1}" for i in range(len(pc1))],
    "PC1 Loading": pc1
})

display(pc1_df)

fig = px.bar(
    pc1_df,
    x="Stock",
    y="PC1 Loading",
    color="PC1 Loading",
    color_continuous_scale="RdBu",
    title="PC1 Loading by Stock"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=500, showlegend=False)
fig.show()

,Stock,PC1 Loading
0,Stock1,0.193084
1,Stock2,0.412581
2,Stock3,0.389177
3,Stock4,0.300268
4,Stock5,0.275968
5,Stock6,0.180660
6,Stock7,0.355114
7,Stock8,0.310125
8,Stock9,0.283571
9,Stock10,0.373340


### (b) Construction of Factor-Neutral Portfolio
We construct a portfolio that is neutral to the first principal component (market factor). This is done by ensuring that the portfolio weights have zero exposure to PC1.

This removes systematic market risk and isolates idiosyncratic returns.

We implement this by adjusting equal weights to eliminate PC1 exposure.

In [54]:
# Step 2: Create factor-neutral weights

n = len(pc1)

# Start with equal weights
w = np.ones(n) / n

# Remove PC1 exposure
w = w - (w @ pc1) * pc1

# Normalize weights
w = w / np.sum(np.abs(w))


weights_df = pd.DataFrame({
    "Stock": [f"Stock{i+1}" for i in range(len(w))],
    "Weight": w
})
print("PC1 exposure:", w @ pc1)

display(weights_df)

fig = px.bar(
    weights_df,
    x="Stock",
    y="Weight",
    color="Weight",
    color_continuous_scale="RdBu",
    title="PC1-Neutral Portfolio Weights"
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=500, showlegend=False)
fig.show()

PC1 exposure: 2.2898349882893854e-16


,Stock,Weight
0,Stock1,0.207535
1,Stock2,-0.136948
2,Stock3,-0.100218
3,Stock4,0.039318
4,Stock5,0.077455
5,Stock6,0.227034
6,Stock7,-0.046758
7,Stock8,0.023848
8,Stock9,0.065523
9,Stock10,-0.075363


### (c) Portfolio Performance Evaluation
We now evaluate the performance of the factor-neutral portfolio on the test dataset.

We compute daily portfolio returns and assess performance using metrics such as average return, volatility, and Sharpe ratio.

In [55]:
# Step 3: Portfolio returns on test data

portfolio_returns = test_returns @ w

mean_return = np.mean(portfolio_returns)
volatility = np.std(portfolio_returns)
sharpe = mean_return / volatility

print("Mean return:", mean_return)
print("Volatility:", volatility)
print("Sharpe ratio:", sharpe)

portfolio_path_df = pd.DataFrame({
    "Day": np.arange(len(portfolio_returns)),
    "Daily Return": portfolio_returns,
    "Cumulative Return": np.cumsum(portfolio_returns)
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Daily portfolio returns", "Cumulative portfolio return")
)

fig.add_trace(
    go.Scatter(
        x=portfolio_path_df["Day"],
        y=portfolio_path_df["Daily Return"],
        mode="lines",
        name="Daily return"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=portfolio_path_df["Day"],
        y=portfolio_path_df["Cumulative Return"],
        mode="lines",
        name="Cumulative return"
    ),
    row=1,
    col=2
)

fig.update_layout(title="Factor-Neutral Portfolio Performance on Test Data", height=500)
fig.update_xaxes(title_text="Test day index")
fig.update_yaxes(title_text="Return")
fig.show()

Mean return: -0.0010460505422177067
Volatility: 0.31875426205001717
Sharpe ratio: -0.0032816833114330756


### (d) Comparison with Equal-Weight Portfolio
To evaluate the impact of factor neutrality, we compare the performance of the factor-neutral portfolio with a simple equal-weight portfolio.

In [56]:
# Equal weight portfolio
w_equal = np.ones(n) / n

returns_equal = test_returns @ w_equal

mean_eq = np.mean(returns_equal)
vol_eq = np.std(returns_equal)
sharpe_eq = mean_eq / vol_eq

print("Equal Weight Sharpe:", sharpe_eq)
print("Factor Neutral Sharpe:", sharpe)

portfolio_compare_df = pd.DataFrame({
    "Portfolio": ["Equal Weight", "PC1 Neutral"],
    "Mean Return": [mean_eq, mean_return],
    "Volatility": [vol_eq, volatility],
    "Sharpe Ratio": [sharpe_eq, sharpe],
})

display(portfolio_compare_df)

compare_long = portfolio_compare_df.melt(
    id_vars="Portfolio",
    value_vars=["Mean Return", "Volatility", "Sharpe Ratio"],
    var_name="Metric",
    value_name="Value"
)

fig = px.bar(
    compare_long,
    x="Metric",
    y="Value",
    color="Portfolio",
    barmode="group",
    title="Equal-Weight vs PC1-Neutral Portfolio Metrics",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=500)
fig.show()

path_compare_df = pd.DataFrame({
    "Day": np.arange(len(returns_equal)),
    "Equal Weight": np.cumsum(returns_equal),
    "PC1 Neutral": np.cumsum(portfolio_returns)
}).melt(id_vars="Day", var_name="Portfolio", value_name="Cumulative Return")

fig = px.line(
    path_compare_df,
    x="Day",
    y="Cumulative Return",
    color="Portfolio",
    title="Cumulative Return: Equal-Weight vs PC1-Neutral Portfolio",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=500)
fig.show()

Equal Weight Sharpe: -0.10639554531947779
Factor Neutral Sharpe: -0.0032816833114330756


,Portfolio,Mean Return,Volatility,Sharpe Ratio
0,Equal Weight,-0.076543,0.719422,-0.106396
1,PC1 Neutral,-0.001046,0.318754,-0.003282


### (e) Transaction Cost Analysis
We incorporate a simple transaction cost model by penalizing large portfolio weights. This reflects the practical cost of implementing the strategy.

In [57]:
# Transaction cost (simple proxy)

cost = 0.001 * np.sum(np.abs(w))

net_return = mean_return - cost

print("Transaction cost:", cost)
print("Net return after cost:", net_return)

tc_bps_grid = np.arange(0, 51, 5)
turnover_proxy = np.sum(np.abs(w))
tc_sensitivity_df = pd.DataFrame({
    "Cost (bps)": tc_bps_grid,
    "Net Mean Return": mean_return - (tc_bps_grid / 10000.0) * turnover_proxy
})

fig = px.line(
    tc_sensitivity_df,
    x="Cost (bps)",
    y="Net Mean Return",
    markers=True,
    title="Transaction-Cost Sensitivity for PC1-Neutral Portfolio"
)
fig.add_hline(y=0, line_dash="dash", annotation_text="break-even")
fig.update_layout(height=500)
fig.show()

Transaction cost: 0.0010000000000000002
Net return after cost: -0.002046050542217707


### Additional portfolio ablation: PC1-neutral vs PC1+PC2-neutral

The base portfolio removes exposure to PC1. As a robustness check, the next cell also removes PC2 exposure and compares the risk/return metrics. This tests whether adding one more hedge actually improves the portfolio or simply over-constrains it.

In [58]:
# Ablation: compare PC1-neutral and PC1+PC2-neutral portfolios.
def neutralize_against_components(base_weights, components):
    """
    Remove exposure to the rows in components from base_weights.
    The final weights are normalized by gross exposure.
    """
    components = np.asarray(components, dtype=float)
    projection = components.T @ np.linalg.pinv(components @ components.T) @ (components @ base_weights)
    w_neutral = base_weights - projection
    gross = np.sum(np.abs(w_neutral))
    if gross > 0:
        w_neutral = w_neutral / gross
    return w_neutral

base_weights = np.ones(n) / n
w_pc1 = neutralize_against_components(base_weights, pca.components_[:1])
w_pc12 = neutralize_against_components(base_weights, pca.components_[:2])

portfolio_ablation_rows = []

for name, weights in [
    ("Equal Weight", base_weights),
    ("PC1 Neutral", w_pc1),
    ("PC1+PC2 Neutral", w_pc12),
]:
    rets = test_returns @ weights
    portfolio_ablation_rows.append({
        "Portfolio": name,
        "PC1 Exposure": float(weights @ pca.components_[0]),
        "PC2 Exposure": float(weights @ pca.components_[1]),
        "Mean Return": float(np.mean(rets)),
        "Volatility": float(np.std(rets)),
        "Sharpe Ratio": float(np.mean(rets) / np.std(rets)),
        "Gross Exposure": float(np.sum(np.abs(weights)))
    })

portfolio_ablation_df = pd.DataFrame(portfolio_ablation_rows)
display(portfolio_ablation_df)

fig = px.bar(
    portfolio_ablation_df.melt(
        id_vars="Portfolio",
        value_vars=["PC1 Exposure", "PC2 Exposure"],
        var_name="Exposure",
        value_name="Value"
    ),
    x="Exposure",
    y="Value",
    color="Portfolio",
    barmode="group",
    title="Factor Exposure Check: Equal Weight vs Neutral Portfolios",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=500)
fig.show()

fig = px.bar(
    portfolio_ablation_df.melt(
        id_vars="Portfolio",
        value_vars=["Volatility", "Sharpe Ratio", "Gross Exposure"],
        var_name="Metric",
        value_name="Value"
    ),
    x="Metric",
    y="Value",
    color="Portfolio",
    barmode="group",
    title="Portfolio Ablation: Does Neutralizing PC2 Help?",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=500)
fig.show()

,Portfolio,PC1 Exposure,PC2 Exposure,Mean Return,Volatility,Sharpe Ratio,Gross Exposure
0,Equal Weight,3.073889e-01,5.744352e-02,-0.076543,0.719422,-0.106396,1.0
1,PC1 Neutral,-4.510281e-17,2.932869e-01,-0.001046,0.318754,-0.003282,1.0
2,PC1+PC2 Neutral,3.191891e-16,3.469447e-17,-0.012916,0.250066,-0.051650,1.0


## 6. Stability Analysis and Rolling PCA

We now analyze the stability of PCA factors over time. Since financial markets are dynamic, it is important to check whether the principal components remain consistent or change over time.

### (a) Split-Half Stability Analysis

We divide the training data into two halves and compute PCA separately on each half. We then compare the first principal components using cosine similarity to measure stability.

In [59]:
from numpy.linalg import norm

# Split training data into two halves
mid = train_returns.shape[0] // 2
X1 = train_returns[:mid]
X2 = train_returns[mid:]

# Fit PCA separately
pca1 = PCA().fit(X1)
pca2 = PCA().fit(X2)

pc1_a = pca1.components_[0]
pc1_b = pca2.components_[0]

# Cosine similarity
cosine_similarity = np.dot(pc1_a, pc1_b) / (norm(pc1_a) * norm(pc1_b))

print("Cosine similarity between PC1s:", cosine_similarity)

split_loading_df = pd.DataFrame({
    "Stock": stock_names,
    "First-half PC1": pc1_a,
    "Second-half PC1": pc1_b
}).melt(id_vars="Stock", var_name="Window", value_name="Loading")

fig = px.bar(
    split_loading_df,
    x="Stock",
    y="Loading",
    color="Window",
    barmode="group",
    title="Split-Half PC1 Loading Comparison",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.add_hline(y=0, line_width=1)
fig.update_layout(height=500)
fig.show()

split_stability_rows = []
for pc_idx in range(min(3, n_assets)):
    v1 = pca1.components_[pc_idx]
    v2 = pca2.components_[pc_idx]
    cos = np.dot(v1, v2) / (norm(v1) * norm(v2))
    split_stability_rows.append({
        "Component": f"PC{pc_idx+1}",
        "Sign-sensitive cosine similarity": cos,
        "Sign-invariant cosine similarity": abs(cos)
    })

split_stability_df = pd.DataFrame(split_stability_rows)
display(split_stability_df)

fig = px.bar(
    split_stability_df,
    x="Component",
    y="Sign-invariant cosine similarity",
    color="Component",
    title="Split-Half Stability: PC1 to PC3",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=450, showlegend=False, yaxis=dict(range=[0, 1.05]))
fig.show()

Cosine similarity between PC1s: 0.9873872980700469


,Component,Sign-sensitive cosine similarity,Sign-invariant cosine similarity
0,PC1,0.987387,0.987387
1,PC2,0.248181,0.248181
2,PC3,0.361375,0.361375


A cosine similarity close to 1 indicates that the first principal component is stable across time. Lower values indicate instability, suggesting that market structure is changing.

### (b) Rolling PCA Analysis

We apply PCA over a rolling window to observe how the first principal component evolves over time.
This helps identify time-varying factor structures.

In [60]:
window = 100
rolling_pc1 = []

for i in range(window, train_returns.shape[0]):
    X_window = train_returns[i-window:i]
    pca_temp = PCA().fit(X_window)
    rolling_pc1.append(pca_temp.components_[0])

# Keep an unaligned copy so the sign-flip issue can be seen directly.
rolling_pc1_raw = np.array(rolling_pc1)
rolling_pc1 = rolling_pc1_raw.copy()

rolling_index = np.arange(window, train_returns.shape[0])
rolling_raw_df = pd.DataFrame(
    rolling_pc1_raw,
    columns=stock_names
)
rolling_raw_df["Window End Day"] = rolling_index

rolling_raw_long = rolling_raw_df.melt(
    id_vars="Window End Day",
    var_name="Stock",
    value_name="Raw PC1 Loading"
)

fig = px.line(
    rolling_raw_long,
    x="Window End Day",
    y="Raw PC1 Loading",
    color="Stock",
    title="Rolling PC1 Loadings Before Sign Alignment",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=550)
fig.show()

### (c) Sign Alignment

Since PCA components can flip signs arbitrarily, we align them to ensure consistency over time.

In [61]:
# Convert to array
rolling_pc1 = np.array(rolling_pc1)

# Align signs
for i in range(1, len(rolling_pc1)):
    if np.dot(rolling_pc1[i], rolling_pc1[i-1]) < 0:
        rolling_pc1[i] = -rolling_pc1[i]

rolling_aligned_df = pd.DataFrame(
    rolling_pc1,
    columns=stock_names
)
rolling_aligned_df["Window End Day"] = rolling_index

rolling_aligned_long = rolling_aligned_df.melt(
    id_vars="Window End Day",
    var_name="Stock",
    value_name="Aligned PC1 Loading"
)

fig = px.line(
    rolling_aligned_long,
    x="Window End Day",
    y="Aligned PC1 Loading",
    color="Stock",
    title="Rolling PC1 Loadings After Sign Alignment",
    color_discrete_sequence=COLOR_SEQUENCE
)
fig.update_layout(height=550)
fig.show()

### (d) Stability Measurement

We compute cosine similarity between consecutive rolling components to quantify stability over time.

In [62]:
similarities = []

for i in range(1, len(rolling_pc1)):
    sim = np.dot(rolling_pc1[i], rolling_pc1[i-1]) / (
        norm(rolling_pc1[i]) * norm(rolling_pc1[i-1])
    )
    similarities.append(sim)

print("Average rolling stability:", np.mean(similarities))

similarity_df = pd.DataFrame({
    "Window End Day": rolling_index[1:],
    "Cosine Similarity": similarities
})

fig = px.line(
    similarity_df,
    x="Window End Day",
    y="Cosine Similarity",
    title="Rolling PCA Stability: Consecutive PC1 Cosine Similarity",
    markers=True
)
fig.add_hline(y=np.mean(similarities), line_dash="dash", annotation_text="average")
fig.update_layout(height=500, yaxis=dict(range=[0, 1.05]))
fig.show()

Average rolling stability: 0.9998245540548157


### (e) Regime Change Interpretation

Periods of low cosine similarity indicate structural changes in the market. These regime shifts imply that PCA factors are not constant and models must adapt over time.

### (f) Walk-Forward Backtesting

We simulate a realistic scenario where PCA is re-estimated over time and used to construct portfolios sequentially.

In [63]:
portfolio_returns_wf = []

window = 100

for i in range(window, len(train_returns)):
    X_window = train_returns[i-window:i]

    pca_temp = PCA().fit(X_window)
    pc1_temp = pca_temp.components_[0]

    # construct neutral weights
    w_temp = np.ones(len(pc1_temp)) / len(pc1_temp)
    w_temp = w_temp - (w_temp @ pc1_temp) * pc1_temp
    w_temp = w_temp / np.sum(np.abs(w_temp))

    # next day return
    r = train_returns[i] @ w_temp
    portfolio_returns_wf.append(r)

print("Walk-forward mean return:", np.mean(portfolio_returns_wf))
print("Walk-forward volatility:", np.std(portfolio_returns_wf))

wf_df = pd.DataFrame({
    "Day": np.arange(window, len(train_returns)),
    "Walk-forward Return": portfolio_returns_wf,
    "Walk-forward Cumulative Return": np.cumsum(portfolio_returns_wf)
})

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Walk-forward daily returns", "Walk-forward cumulative return")
)

fig.add_trace(
    go.Scatter(
        x=wf_df["Day"],
        y=wf_df["Walk-forward Return"],
        mode="lines",
        name="Daily return"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=wf_df["Day"],
        y=wf_df["Walk-forward Cumulative Return"],
        mode="lines",
        name="Cumulative return"
    ),
    row=1,
    col=2
)

fig.update_layout(title="Walk-Forward Factor-Neutral Backtest", height=500)
fig.update_xaxes(title_text="Training day index")
fig.update_yaxes(title_text="Return")
fig.show()

Walk-forward mean return: -0.009612902065217558
Walk-forward volatility: 0.3221458547425612


The rolling PCA results show that the first principal component is fairly persistent, but not fixed. Sign alignment is necessary before interpreting the loading paths; without it, normal eigenvector sign flips can look like regime changes.

The portfolio results also show why PCA hedges need to be treated as live estimates rather than permanent constants. A hedge built from an old PCA basis can remove the original dominant factor, but its exposure can drift as the covariance structure changes. In practice, this argues for periodic recalibration and for checking transaction costs before adding extra neutrality constraints.

---

## Final evaluator-facing takeaways

1. **Returns, not prices, carry the factor structure.** Prices drift and compound, while log returns isolate the daily co-movement needed for PCA.

2. **Standardization changes the question.** PCA on standardized returns finds common correlation structure; PCA on raw returns gives more influence to high-volatility stocks. Both are useful, but they answer different questions.

3. **Dimensionality reduction is useful but lossy.** The first few PCs capture most systematic structure, but the reconstruction-error and random-subspace ablation show that the discarded components still contain idiosyncratic stock-level information.

4. **Information leakage is not solved by saying "unsupervised."** PCA fitted on train + test can rotate toward future covariance structure. That makes live residual risk look smaller than it would have looked using only information available at the time.

5. **Factor-neutral portfolios are constraint-sensitive.** Neutralizing more PCs can reduce factor exposure, but it can also increase implementation complexity and turnover. The right hedge is not the one with the most constraints; it is the one that improves risk after transaction costs.